In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)
 

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [2]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [3]:
# 2. 불균형 확인
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [4]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [5]:
# 3. 이상값 탐지(var3의 min: -999999)
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [6]:
# 4-1. 전처리(var3의 -999999 -> 2  & ID 제거)
cust_df["var3"] = cust_df["var3"].replace(-999999,2)
cust_df.drop("ID", axis=1, inplace=True)

In [7]:
# 일반 데이터와 레이블 분리
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]

In [8]:
# 4-2. 전처리(분산 0인 컬럼 제거)
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [9]:
# 값이 완전히 동일한 컬럼 중 뒤에 나온 것들의 이름을 반환
import numpy as np

def find_duplicate_columns(df):

    groups = {}
    for col in df.columns:
        v = df[col].values
        # 지문(sum/min/max)으로 후보를 먼저 좁힘
        key = (v.sum(), v.min(), v.max())
        groups.setdefault(key, []).append(col)

    dup = set()
    for cols in groups.values():
        if len(cols) < 2:
            continue
        for i in range(len(cols)):
            if cols[i] in dup:
                continue
            for j in range(i + 1, len(cols)):
                if cols[j] in dup:
                    continue
                if np.array_equal(df[cols[i]].values, df[cols[j]].values):
                    dup.add(cols[j])
    return sorted(dup)


dup_cols = find_duplicate_columns(X_features_clean)
print(f"중복 컬럼 개수: {len(dup_cols)}")
print(f"삭제할 컬럼들: {dup_cols}")

X_features_clean = X_features_clean.drop(columns=dup_cols)
print(f"정제 후 피처 shape: {X_features_clean.shape}")

중복 컬럼 개수: 29
삭제할 컬럼들: ['delta_num_reemb_var13_1y3', 'delta_num_reemb_var17_1y3', 'delta_num_reemb_var33_1y3', 'delta_num_trasp_var17_in_1y3', 'delta_num_trasp_var17_out_1y3', 'delta_num_trasp_var33_in_1y3', 'delta_num_trasp_var33_out_1y3', 'ind_var13_medio', 'ind_var18', 'ind_var25', 'ind_var26', 'ind_var29', 'ind_var29_0', 'ind_var32', 'ind_var34', 'ind_var37', 'ind_var39', 'num_var13_medio', 'num_var18', 'num_var25', 'num_var26', 'num_var29', 'num_var29_0', 'num_var32', 'num_var34', 'num_var37', 'num_var39', 'saldo_medio_var13_medio_ult1', 'saldo_var29']
정제 후 피처 shape: (76020, 306)


In [10]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 수정된 get_clf_eval() 함수
def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score

RANDOM_STATE = 42


# def preprocess(df):
#     """최종 채택된 1~5단계 전처리만 적용 (369 -> 145개 피처)."""
#     y = df['TARGET']
#     X = df.drop(columns=['ID', 'TARGET'])

#     dup_cols = X.columns[X.T.duplicated()].tolist()
#     X = X.drop(columns=dup_cols)

#     stds = X.std()
#     X = X.drop(columns=stds[stds == 0].index.tolist())

#     num_rows = X.shape[0]
#     sparse_cols = [c for c in X.columns if (X[c] == 0).sum() / num_rows >= 0.99]
#     X = X.drop(columns=sparse_cols)

#     X['var38'] = np.log1p(X['var38'])
#     X['var15_below_23'] = (X['var15'] < 23).astype(int)
#     X['var15_bin'] = pd.cut(X['var15'], bins=5, labels=False).astype(int)

#     return X, y


# # 원본 CSV부터 새로 읽어 재구성 (노트북 앞부분 cust_df 상태와 무관하게 독립적으로 동작)
# raw_df = pd.read_csv('../data/santander-customer-satisfaction/train.csv', encoding='latin-1')
# raw_df['var3'] = raw_df['var3'].replace(-999999, 2)

# X, y = preprocess(raw_df)
# print(f"최종 피처 수: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X_features_clean, y_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=y_labels
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

final_clf = LogisticRegression(
    class_weight='balanced', C=1.0, penalty='l2',
    solver='liblinear', max_iter=3000, random_state=RANDOM_STATE,
)
final_clf.fit(X_train_scaled, y_train)

test_proba = final_clf.predict_proba(X_test_scaled)[:, 1]
test_pred_05 = final_clf.predict(X_test_scaled)

print("\n=== threshold 0.5 (기본) ===")
print(f"accuracy={accuracy_score(y_test, test_pred_05):.4f}  "
      f"roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_05):.4f}  "
      f"precision={precision_score(y_test, test_pred_05):.4f}")

get_clf_eval(y_test, test_pred_05, test_proba)


# StandardScaler 전
# === threshold 0.5 (기본) ===
# accuracy=0.9044  roc_auc=0.6751  recall=0.1096  precision=0.0671

# StandardScaler 후
# === threshold 0.5 (기본) ===
# accuracy=0.6847  roc_auc=0.8032  recall=0.7691  precision=0.0905


=== threshold 0.5 (기본) ===
accuracy=0.6847  roc_auc=0.8032  recall=0.7691  precision=0.0905
오차 행렬
[[9947 4655]
 [ 139  463]]
정확도: 0.6847, 정밀도: 0.0905, 재현율: 0.7691,    F1: 0.1619, AUC:0.8032


In [12]:
# 학습 데이터 안에서 내부 검증셋을 분리해 임계값을 고른다 (테스트셋 누수 방지)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)

scaler_val = StandardScaler()
X_tr2_scaled = pd.DataFrame(scaler_val.fit_transform(X_tr2), columns=X_tr2.columns)
X_val_scaled = pd.DataFrame(scaler_val.transform(X_val), columns=X_val.columns)

clf_val = LogisticRegression(
    class_weight='balanced', C=1.0, penalty='l2',
    solver='liblinear', max_iter=3000, random_state=RANDOM_STATE,
)
clf_val.fit(X_tr2_scaled, y_tr2)
val_proba = clf_val.predict_proba(X_val_scaled)[:, 1]

chosen_threshold = None
for t in np.arange(0.50, 0.03, -0.01):
    val_pred = (val_proba >= t).astype(int)
    if recall_score(y_val, val_pred) >= 0.80:
        chosen_threshold = t
        break
if chosen_threshold is None:
    chosen_threshold = 0.03

print(f"선택된 threshold (val recall >= 0.80을 만족하는 가장 높은 값): {chosen_threshold:.2f}")

# 원래 (전체 X_train으로 학습한) final_clf의 test_proba에 새 threshold 적용
test_pred_tuned = (test_proba >= chosen_threshold).astype(int)

print("\n=== threshold 0.5 (before) vs {:.2f} (after) ===".format(chosen_threshold))
print(f"[0.50] accuracy={accuracy_score(y_test, test_pred_05):.4f}  roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_05):.4f}  precision={precision_score(y_test, test_pred_05):.4f}")
print(f"[{chosen_threshold:.2f}] accuracy={accuracy_score(y_test, test_pred_tuned):.4f}  roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_tuned):.4f}  precision={precision_score(y_test, test_pred_tuned):.4f}")

get_clf_eval(y_test, test_pred_tuned, test_proba)


# 내 전처리
# 선택된 threshold (val recall >= 0.80을 만족하는 가장 높은 값): 0.41
# === threshold 0.5 (before) vs 0.41 (after) ===
# [0.50] accuracy=0.6847  roc_auc=0.8032  recall=0.7691  precision=0.0905
# [0.41] accuracy=0.6191  roc_auc=0.8032  recall=0.8206  precision=0.0800

# 성준님 전처리
# threshold	accuracy	roc_auc	recall	precision
# 0.50 (before)	0.7459	0.8116	0.7359	0.1068
# 0.40 (after, val recall≥0.80 기준 선택)	0.6190	0.8116	0.8239	0.0802

# 성준님 roc_auc가 더 나은 결과. 성준님 정제가 더 효과적.(희소행렬 제거, 로그화 추가.)

선택된 threshold (val recall >= 0.80을 만족하는 가장 높은 값): 0.41

=== threshold 0.5 (before) vs 0.41 (after) ===
[0.50] accuracy=0.6847  roc_auc=0.8032  recall=0.7691  precision=0.0905
[0.41] accuracy=0.6191  roc_auc=0.8032  recall=0.8206  precision=0.0800
오차 행렬
[[8919 5683]
 [ 108  494]]
정확도: 0.6191, 정밀도: 0.0800, 재현율: 0.8206,    F1: 0.1457, AUC:0.8032
